In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
import numpy as np
from tqdm import tqdm
from datetime import datetime
import gc
import random

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Используется устройство: {device}")
print(f"Поддержка Tensor Cores: {'Да' if torch.cuda.get_device_capability(0)[0] >= 7 else 'Нет'}")

Используется устройство: cuda:0
Поддержка Tensor Cores: Да


In [2]:
# фиксируем сиды для воспроизводимости

seed = 1443
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Данные

In [3]:
# Загружаем датасет
trainset = load_dataset("phystech/cifar_train", token=None, split='train')
trainset = [(x['data'], x['label']) for x in trainset]

# Делим датасет на train / test
trainset, testset = trainset[:-1_000], trainset[-1_000:]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [4]:
class CustomDataset(Dataset):

    def __init__(self, data):
        self.data = data

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        feature = torch.tensor(self.data[idx][0], dtype=torch.float32)
        target = torch.tensor(self.data[idx][1], dtype=torch.int64)

        return feature, target


train_data = CustomDataset(trainset)
train_loader = DataLoader(
    train_data,
    batch_size=7_000, # большой батч для демонстрации
    shuffle=True, # перемешивание данных
    drop_last=True, # удаление неполного батча
    num_workers=2  # для параллельной загрузки
)

test_data = CustomDataset(testset)
test_loader = DataLoader(
    test_data,
    batch_size=32,
    shuffle=False,
    num_workers=2
)


In [5]:
next(iter(test_loader))

[tensor([[ 0.1918,  0.5014,  3.5726,  ...,  2.8769, -1.1636, -0.1175],
         [ 0.4161, -0.7559,  0.4905,  ..., -2.2118, -0.1748,  0.3553],
         [-0.5362, -0.7087, -0.0222,  ...,  0.7959, -0.2765,  0.7136],
         ...,
         [-1.2581,  1.0738, -2.4495,  ..., -0.2814,  0.1745,  0.0518],
         [-0.2965,  1.6405,  0.5294,  ...,  0.3857, -0.4691,  0.2069],
         [ 1.1016,  1.5751, -1.7935,  ...,  0.4650,  0.9367, -0.3583]]),
 tensor([5, 3, 4, 2, 0, 8, 0, 5, 1, 6, 0, 0, 2, 4, 4, 4, 5, 9, 2, 6, 6, 7, 2, 0,
         1, 7, 5, 9, 6, 7, 3, 2])]

## Инициализируем модель

In [6]:
model = nn.Sequential(
    nn.BatchNorm1d(768),
    nn.Dropout(0.1),
    nn.Linear(768, 128),
    nn.ReLU(),
    nn.BatchNorm1d(128),
    nn.Dropout(0.1),
    nn.Linear(128, 10)
)
model

Sequential(
  (0): BatchNorm1d(768, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (1): Dropout(p=0.1, inplace=False)
  (2): Linear(in_features=768, out_features=128, bias=True)
  (3): ReLU()
  (4): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (5): Dropout(p=0.1, inplace=False)
  (6): Linear(in_features=128, out_features=10, bias=True)
)

##  Обучаем модель

In [8]:
# очищаем память графического процессора
for var_name in ['model', 'inputs', 'labels', 'outputs', 'optimizer']:
    if var_name in globals():
        del globals()[var_name]
gc.collect()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [9]:
# Возьмем модель побольше для демонстрации
model = nn.Sequential(
    nn.Linear(768, 10_000),
    nn.ReLU(),
    nn.Linear(10_000, 10_000),
    nn.ReLU(),
    nn.Linear(10_000, 10)
)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

model.to(device)
model.train()
running_loss = np.inf

start = datetime.now()
for epoch in range(10):
    loss_value = 0
    total = 0
    for inputs, labels in tqdm(train_loader, desc=f'epoch: {epoch} loss: {running_loss:.4f}'):
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_value += (loss.item() * labels.size(0))
        total += labels.size(0)
        running_loss = loss_value/total


print(f'\nВремя обучения: {(datetime.now() - start).total_seconds():.0f} s')


epoch: 9 loss: 0.3518: 100%|██████████| 7/7 [00:11<00:00,  1.69s/it]


Время обучения: 104 s


### Оцениваем модель

In [10]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)


        outputs = model(inputs)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

print('Test Accuracy: ', 100 * correct / total)

Test Accuracy:  78.7


## Обучаем модель с Mixed Precision

In [11]:
# очищаем память графического процессора
for var_name in ['model', 'inputs', 'labels', 'outputs', 'optimizer']:
    if var_name in globals():
        del globals()[var_name]
gc.collect()

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()

In [12]:
# Возьмем модель побольше
model = nn.Sequential(
    nn.Linear(768, 10_000),
    nn.ReLU(),
    nn.Linear(10_000, 10_000),
    nn.ReLU(),
    nn.Linear(10_000, 10)
)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scaler = torch.amp.GradScaler()

model.to(device)
model.train()
running_loss = np.inf

start = datetime.now()
for epoch in range(10):
    loss_value = 0
    total = 0
    for inputs, labels in tqdm(train_loader, desc=f'epoch: {epoch} loss: {running_loss:.4f}'):
        inputs, labels = inputs.to(device), labels.to(device)

        with torch.amp.autocast('cuda'):
            outputs = model(inputs)
            loss = criterion(outputs, labels)

        optimizer.zero_grad()
        scaler.scale(loss).backward() # умножает лосс на большое число и делает backward проход
        scaler.step(optimizer) # проверяет не превратились ли градиенты в наны и делает шаг с учетом масштабированных градентов
        scaler.update() # если градиенты превратились в наны, то уменьшает коэффициент масштабирования лосса

        loss_value += (loss.item() * labels.size(0))
        total += labels.size(0)
        running_loss = loss_value/total

print(f'\nВремя обучения: {(datetime.now() - start).total_seconds():.0f} s')

epoch: 9 loss: 0.3733: 100%|██████████| 7/7 [00:04<00:00,  1.61it/s]


Время обучения: 46 s


In [13]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)


        outputs = model(inputs)
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

print('Test Accuracy: ', 100 * correct / total)

Test Accuracy:  78.1
